# CarePath — 03 Grounded SOAP training `[L4/A100]`

Call the sibling SOAP orchestrator. Its data, factual filters, and export gates remain owned by the SOAP training package.

In [ ]:
# CarePath bootstrap: local checkout first, otherwise a private GitHub clone.
import importlib.util
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

def _find(start):
    for directory in [start, *start.parents]:
        if (directory / 'pyproject.toml').exists() and (directory / 'scribe' / 'carepath').exists():
            return directory
    return None

def _secret():
    for key in ('CAREPATH_GITHUB_TOKEN', 'GITHUB_TOKEN'):
        if os.environ.get(key):
            return os.environ[key]
    try:
        from google.colab import userdata
        for key in ('CAREPATH_GITHUB_TOKEN', 'GITHUB_TOKEN'):
            try:
                value = userdata.get(key)
                if value:
                    return value
            except Exception:
                pass
    except Exception:
        pass
    return None

def _inject_runtime_secrets():
    """Copy only approved named Colab secrets into memory for child processes."""
    try:
        from google.colab import userdata
    except Exception:
        return
    for key in ('LLM_API_KEY', 'HF_TOKEN'):
        if os.environ.get(key):
            continue
        try:
            value = userdata.get(key)
        except Exception:
            value = None
        if value:
            os.environ[key] = value

REPO = _find(Path.cwd().resolve())
if REPO is None and importlib.util.find_spec('google.colab'):
    target = Path('/content/carepath')
    REPO = _find(target)
    if REPO is None:
        if target.exists():
            shutil.rmtree(target)
        url = os.environ.get('CAREPATH_REPO_URL', 'https://github.com/truong-tt/carepath.git')
        if '://' in url and '@' in url.split('://', 1)[1].split('/', 1)[0]:
            raise SystemExit('CAREPATH_REPO_URL must not contain credentials; use a Colab Secret.')
        token = _secret()
        clone_env = dict(os.environ)
        clone_env['GIT_TERMINAL_PROMPT'] = '0'
        with tempfile.TemporaryDirectory(prefix='carepath_git_') as temp:
            if token:
                askpass = Path(temp) / 'askpass.sh'
                askpass.write_text(
                    '#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n',
                    encoding='utf-8',
                )
                askpass.chmod(0o700)
                clone_env['GIT_ASKPASS'] = str(askpass)
                clone_env['GITHUB_TOKEN'] = token
            result = subprocess.run(
                ['git', 'clone', url, str(target)],
                env=clone_env,
                capture_output=True,
                text=True,
            )
        if result.returncode:
            error = result.stderr or result.stdout
            if token:
                error = error.replace(token, '***')
            raise SystemExit(
                'Clone failed. Add a Colab Secret named GITHUB_TOKEN with repository read access, '
                'enable Notebook access, then rerun.\n' + error
            )
        REPO = target

assert REPO, 'Run the notebook inside CarePath or from Google Colab.'
os.chdir(REPO)
sys.path[:0] = [str(REPO / 'scribe' / 'training'), str(REPO / 'scribe')]
_inject_runtime_secrets()

PROFILE = os.environ.get('CAREPATH_PROFILE', 'smoke')
CONFIRM_PAID = os.environ.get('CAREPATH_CONFIRM_PAID') == '1'
from gec.notebook import init_stage
CTX = init_stage(PROFILE, confirm_paid=CONFIRM_PAID)
P, PROF = CTX.paths, CTX.profile


In [ ]:
# Pin the repository's tested training and combined-app environment.
extras = ['dev', 'training']
if PROFILE == 'reproduction':
    extras.append('training-tts')
if PROF.paid:
    extras.append('training-fast')
extra = '.[' + ','.join(extras) + ']'
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', extra, '-e', './shared', '-e', './interpreter'],
    check=True,
)


In [ ]:
CTX.run_soap_pipeline()
